## Convert a Mono Recording to Stereo

### Description

Given an audio source and the source's angle in time with respect to an observer, I calculate the stereo audio that reflects that path.

For a slow moving source, this is achieved with time delays, simply measuring the distance from the source to each ear in time. In this implementation I don't worry about clutter or multipath considerations as I would in something like a radar simulator. The delays and path loss attenuations are applied in the frequency domain with an overlap-add method. I use a window size of 1024 samples, or 23ms. Because the time delay is calculated and applied at such a quick rate, the doppler effect can be heard in the stereo recordings for moving sources. 

To account for the human head, in order to model sounds from behind a listener I need to apply some kind of EQ effect. Generally sounds from behind us are low passed above about 1.5kHz, so I impelment that in the frequency domain as well.

To be incredibly precise, I could just multiply by the Head-related Transfer Function, a standard that includes phase delays for binaurial perception and magnitude adjustments tuned to the geometry of the human head. It's just too simple of an implementation and I don't get much experience out of it.

### Imports and Function Definitions

In [481]:
import numpy as np
from scipy.io.wavfile import read
import IPython.display as ipd

In [423]:
speed_of_sound = 343 # meters per second, constant for audible frequencies
fsamp = 44100 # Hz
head_width = 0.15 # meters

In [470]:
# Free space path loss for audible frequencies
def path_loss_dB(distance, freqs):
    speed_of_sound = 343 # meters per second, constant for audible frequencies
    return 20*np.log10(max(distance,0.1)) + 20*np.log10([max(f,1) for f in freqs]) + 20 * np.log10(4*np.pi/speed_of_sound)

In [467]:
# Assume the listener is at the origin
# Distance measured in meters
# Distance between the ears is about 0.15 meters
def mono_to_stereo(audio, rate, init_coords=[0,0], velocity=[0,0]):
    # If stereo convert to mono
    if audio.ndim > 1:
        audio = audio.mean(axis=1).astype(audio.dtype)
    N = len(audio)
    n_fft = 1024
    time = np.arange(0, (N+n_fft)/rate, 1/rate) # seconds
    
    # Calculate source positions and distance over time
    x_coords = init_coords[0] + velocity[0] * time
    y_coords = init_coords[1] + velocity[1] * time
    distances_left = np.sqrt((x_coords + head_width / 2)**2 + y_coords**2)
    distances_right = np.sqrt((x_coords - head_width / 2)**2 + y_coords**2)
    
    # Calculate delays to each ear
    delays_left = distances_left / speed_of_sound
    delays_right = distances_right / speed_of_sound

    # Apply frequency-dependent path loss in the frequency domain
    window = np.hanning(n_fft)
    output_left = np.zeros(len(time))
    output_right = np.zeros(len(time))
    hop_length = n_fft // 4

    # Specify frequency domain coefficients for a low pass filter to apply when sounds are behind the listener
    freqs = np.fft.rfftfreq(n_fft, d=1/rate)
    start_decay_idx = int(1000 * (len(freqs) - 1) / (rate/2)) # The human hearing low pass filter takes noticeable effect around 1.5kHz
    end_decay_idx = int(3000 * (len(freqs) - 1) / (rate/2))
    low_pass = np.zeros_like(freqs)
    low_pass[:start_decay_idx] = 1
    low_pass[start_decay_idx:end_decay_idx] = [1 - (i-start_decay_idx)/(end_decay_idx - start_decay_idx) for i in range(start_decay_idx, end_decay_idx)]

    # Apply the delays and attenuations in the frequency domain
    for start_idx in range(0, N - n_fft, hop_length):
        # Calculate FFT of input block
        block = audio[start_idx:start_idx + n_fft] * window
        X = np.fft.rfft(block)

        # Apply delay via phase shift
        X_left = X * np.exp(-1j * 2 * np.pi * freqs * delays_left[start_idx]) 
        X_right = X * np.exp(-1j * 2 * np.pi * freqs * delays_right[start_idx])

        # Apply path loss
        X_left = X_left * 10**(-(path_loss_dB(distances_left[start_idx], freqs) / 20))
        X_right = X_right * 10**(-(path_loss_dB(distances_right[start_idx], freqs) / 20))

        # Apply a low pass filter if the sound is behind the listener
        if y_coords[start_idx] < 0:
            X_left *= low_pass
            X_right *= low_pass

        # Recontruct in the time domain
        block_left = np.fft.irfft(X_left) * window 
        block_right = np.fft.irfft(X_right) * window 

        output_left[start_idx:start_idx + n_fft] += block_left
        output_right[start_idx:start_idx + n_fft] += block_right

    return [output_left, output_right]

### Example Usage

In [439]:
# Load audio clip
rate, input_guitar = read("../clips/mine/fairest_of_the_seasons.wav")
dur = 10 # seconds
input_guitar = input_guitar[:dur * rate]

# Convert to mono if stereo
if input_guitar.ndim > 1:
    input_guitar = input_guitar.mean(axis=1).astype(input_guitar.dtype)

# Convert to floating point and normalize
input_guitar = input_guitar.astype(np.float32) / np.max(np.abs(input_guitar))

# Play original
ipd.display(ipd.Audio(input_guitar, rate=rate))

/var/folders/yz/xcf27lrx69l4v_mqmrv7_z_80000gn/T/ipykernel_19805/510926793.py:2: WavFileWarning: Chunk (non-data) not understood, skipping it.
  rate, input_guitar = read("../clips/mine/fairest_of_the_seasons.wav")


In [479]:
stereo_guitar = mono_to_stereo(input_guitar, rate, init_coords=[-4,1], velocity=[1,0])

In [480]:
# Play audio
ipd.display(ipd.Audio(stereo_guitar, rate=rate, normalize=False))